# Are your two datasets actually independent?

**A test you can run on any two datasets before you trust a reconciliation
between them.**

This notebook grew out of a mistake. We benchmarked a facility matcher by
reconciling Nigeria's GRID3 reference data against OpenStreetMap, called the
result independent validation, and published a precision figure. It was not
independent validation. Two lines of arithmetic would have told us that before
we wrote a word.

The test is simple enough to run on anything, and this notebook is mostly about
teaching it. The facility matching is the worked example.

**Runtime:** a few minutes. The Overture section needs network; everything else
runs on data in this repository.

---

## The problem with "we validated against an independent source"

When you reconcile two datasets and get a high match rate, there are two
explanations:

1. Your matcher is good.
2. One dataset is downstream of the other, and you are measuring a copy.

These look identical in a precision score. They are not remotely the same
thing, and the second one is worthless as validation — you have confirmed that
a copy resembles its original.

## The test

Take the pairs your matcher agreed on and look at the **distance between their
coordinates**.

Independent field surveys of the same building do not agree exactly. Handheld
GPS error is 3 to 10 metres in good conditions, worse under tree cover or
beside a wall. Two teams, two devices, two visits: you expect tens of metres of
disagreement, distributed.

So:

- **A median distance near zero, and a large fraction at *exactly* zero, means
  one dataset was derived from the other.** Real independent observation cannot
  produce that.
- **A median in the tens of metres, with few exact ties, is what independence
  looks like.**

That is the whole test. It takes about ten lines.

## Step 1 — Load three sources

`GRID3` is the reference dataset under test. We will check it against two
others that are both described as independent.

In [1]:
import csv, warnings, statistics, collections, json
warnings.filterwarnings("ignore")

HEALTH = ("hospital", "clinic", "doctor", "health", "medical", "pharmacy", "dentist")

with open("../../data/GRID3_NGA_health_facilities_v2.csv", encoding="utf-8-sig") as fh:
    grid3 = [r for r in csv.DictReader(fh) if r["state"] == "Kano"]

with open("../../data/osm_kano.csv", encoding="utf-8-sig") as fh:
    osm = [r for r in csv.DictReader(fh) if r.get("name", "").strip()]

GRID3 = [{"name": r["facility_name"], "lat": r["latitude"], "lon": r["longitude"],
          "type": r["facility_level_option"]} for r in grid3]
OSM = [{"name": r["name"], "lat": r["lat"], "lon": r["lon"],
        "type": r.get("amenity", "")} for r in osm]

print(f"GRID3 (Kano)      : {len(GRID3):,} facilities")
print(f"OpenStreetMap     : {len(OSM):,} named features")

GRID3 (Kano)      : 1,723 facilities
OpenStreetMap     : 685 named features


## Step 2 — Reconcile, and look at the distances

`crosswalk` returns matched pairs with the distance between their coordinates
already in the evidence.

In [2]:
from arche.resolve import crosswalk

def independence_report(label, A, B):
    """The whole test. Run this on any two datasets you are told are independent."""
    result = crosswalk(A, B, entity="place")
    matched = [m for m in result["matches"] if m["decision"] == "match"]
    d = sorted(m["evidence"].get("distance_km", 0.0) for m in matched)
    if not d:
        print(f"{label}: no matches to judge")
        return
    exact = sum(1 for x in d if x == 0.0)
    covered = len({m["a_id"] for m in matched})
    print(f"{label}")
    print(f"   records                 : {len(A):,}")
    print(f"   matched                 : {covered:,}  ({100*covered/len(A):.1f}% coverage)")
    print(f"   median distance         : {statistics.median(d):.3f} km")
    print(f"   90th percentile         : {d[int(0.9*len(d))]:.3f} km")
    print(f"   at EXACTLY 0.00 km      : {exact}  ({100*exact/len(d):.0f}% of matches)")
    print()

independence_report("GRID3 x OpenStreetMap", OSM, GRID3)

GRID3 x OpenStreetMap
   records                 : 685
   matched                 : 521  (76.1% coverage)
   median distance         : 0.000 km
   90th percentile         : 0.050 km
   at EXACTLY 0.00 km      : 319  (59% of matches)



**Read that carefully.** A median of 0.000 km, and the majority of matches
at exactly zero.

There is no way two independent surveys produce that. It means the OSM health
facilities for this state were imported from the same lineage as GRID3 — most
likely the Nigeria Health Facility Registry, which GRID3 also draws on.

We had already found that 68% of GRID3's national records carry
`facility_name_source: NHFR_2024`, and had used that to argue GRID3 should not
be validated against the HDX registry mirror. Then we validated it against OSM
instead and did not apply the same scrutiny to our own choice.

Any precision figure from this pairing is a **consistency check**, not
validation. Worth having. Not what we called it.

## Step 3 — A source with published lineage

Overture Maps publishes the provenance of every feature, so independence does
not have to be inferred. This section needs network and DuckDB.

In [3]:
import duckdb

BBOX = (7.4, 10.3, 9.6, 12.7)          # Kano State, generous
RELEASE = "2026-07-22.0"

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial; INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='us-west-2';")

path = f"s3://overturemaps-us-west-2/release/{RELEASE}/theme=places/type=place/*"
rows = con.execute(f"""
    SELECT names.primary AS name,
           categories.primary AS category,
           ROUND(ST_Y(geometry), 6) AS lat,
           ROUND(ST_X(geometry), 6) AS lon,
           sources
    FROM read_parquet('{path}', filename=true, hive_partitioning=1)
    WHERE bbox.xmin BETWEEN {BBOX[0]} AND {BBOX[2]}
      AND bbox.ymin BETWEEN {BBOX[1]} AND {BBOX[3]}
      AND names.primary IS NOT NULL
""").fetchall()

print(f"pulled {len(rows):,} Overture places in the Kano bbox")

pulled 7,345 Overture places in the Kano bbox


### The lineage is in the data

This is the part worth copying. Overture records which dataset each feature
came from, so you can check independence directly instead of inferring it from
coordinates.

In [4]:
health = [r for r in rows
          if r[1] and any(k in r[1] for k in HEALTH) and (r[0] or "").strip()]

lineage = collections.Counter()
for r in health:
    for s in (r[4] or []):
        lineage[s.get("dataset", "?")] += 1

print(f"Overture health places in Kano: {len(health)}")
print()
print("source lineage:")
for dataset, n in lineage.most_common():
    print(f"   {dataset:16} {n:,}")

Overture health places in Kano: 358

source lineage:
   Overture         358
   meta             354
   Microsoft        3
   Foursquare       1


**Meta, Microsoft, Foursquare. No OpenStreetMap, no GRID3, no NHFR.**

That is a genuinely independent compilation of the same territory, and it is
stated in the data rather than assumed.

## Step 4 — What independence actually looks like

Run the same test on the independent source and compare the shapes.

In [5]:
OVERTURE = [{"name": r[0], "lat": r[2], "lon": r[3], "type": r[1]} for r in health]

independence_report("GRID3 x OpenStreetMap", OSM, GRID3)
independence_report("GRID3 x Overture (Meta)", OVERTURE, GRID3)

GRID3 x OpenStreetMap
   records                 : 685
   matched                 : 521  (76.1% coverage)
   median distance         : 0.000 km
   90th percentile         : 0.050 km
   at EXACTLY 0.00 km      : 319  (59% of matches)



GRID3 x Overture (Meta)
   records                 : 358
   matched                 : 37  (10.3% coverage)
   median distance         : 0.050 km
   90th percentile         : 0.750 km
   at EXACTLY 0.00 km      : 3  (8% of matches)



Two completely different signatures:

| | OSM | Overture |
|---|---|---|
| coverage | ~76% | ~10% |
| median distance | 0.000 km | ~0.05 km |
| at exactly 0.00 km | ~59% | ~8% |

The independent source agrees on **fewer** facilities and agrees **less
precisely** about where they are. That is not the matcher performing worse. It
is what honest disagreement between two observations looks like.

### The coverage gap is a finding, not a defect

Overture matches only about a tenth of its health places to GRID3. That is
because they are different populations: GRID3 is overwhelmingly rural primary
health centres and health posts, while Meta's place data covers named, signed,
commercial facilities — urban hospitals and clinics.

An independent source can validate the urban tier and says almost nothing about
the rural network, which is exactly where a national facility list matters
most. Knowing that is more useful than a precision score.

## Step 5 — Applying this to your own data

The test is short enough to inline. If you have two datasets and a matcher:

In [6]:
def looks_derived(distances_km, exact_threshold=0.25, median_threshold=0.005):
    """Heuristic: does this pairing look like a copy rather than two surveys?

    `distances_km` are the coordinate distances of matched pairs.

    Not a proof. A deliberately blunt instrument that catches the case where
    someone hands you "an independent source" that is a re-publication.
    """
    if not distances_km:
        return None
    exact = sum(1 for d in distances_km if d == 0.0) / len(distances_km)
    med = statistics.median(distances_km)
    return {
        "fraction_exact": round(exact, 3),
        "median_km": round(med, 4),
        "verdict": ("likely derived" if exact > exact_threshold and med < median_threshold
                    else "consistent with independence"),
    }

for label, A in [("OpenStreetMap", OSM), ("Overture", OVERTURE)]:
    res = crosswalk(A, GRID3, entity="place")
    d = [m["evidence"].get("distance_km", 0.0)
         for m in res["matches"] if m["decision"] == "match"]
    print(f"{label:16} {looks_derived(d)}")

OpenStreetMap    {'fraction_exact': 0.585, 'median_km': 0.0, 'verdict': 'likely derived'}


Overture         {'fraction_exact': 0.081, 'median_km': 0.05, 'verdict': 'consistent with independence'}


### Caveats, because this is a blunt instrument

- **Rounding matters.** `distance_km` here is rounded to 2 decimal places, so
  anything under about 5 metres reads as exactly zero. Tighten it if your
  coordinates are more precise.
- **Some exact ties are legitimate.** A facility geocoded to the same postcode
  centroid by both sources will agree exactly without either copying the other.
  Look at the *fraction*, not individual pairs.
- **It detects derivation, not quality.** A derived source can still be useful;
  it just cannot validate its own ancestor.
- **Low coverage is not evidence of independence** on its own. A bad matcher
  also produces low coverage. Read coverage and distance together.

## What to take away

1. **Ask where a dataset came from before you validate against it.** Overture
   publishes lineage. Most sources do not, and you have to infer.
2. **The coordinate-distance distribution is a cheap lineage test.** Median
   near zero plus a large exact-tie fraction means derivation.
3. **Independent sources disagree, and that is the point.** If your validation
   source agrees with you perfectly, it is probably not a source.
4. **Apply the scrutiny to your own choices.** We caught the GRID3/HDX
   circularity and then walked into a softer version of it. The test that
   catches other people is the test worth running on yourself.

## Next

- `01_facility_reconciliation.ipynb` — the crosswalk this benchmark measures
- `03_llm_vs_arche.ipynb` — where a language model helps and where it does not